In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
import numpy as np

In [ ]:
(x_train_official, y_train_official), (x_test_official, y_test_official) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [ ]:
x_all = np.concatenate([x_train_official, x_test_official])
y_all = np.concatenate([y_train_official, y_test_official])

In [ ]:
x_all = x_all.astype('float32') / 255.0

In [ ]:
num_classes = 10
y_all = tf.keras.utils.to_categorical(y_all, num_classes)

In [ ]:
TOTAL_IMAGES = 60000
TRAIN_SIZE = 48000
VAL_SIZE = 6000

In [ ]:
x_train, y_train = x_all[:TRAIN_SIZE], y_all[:TRAIN_SIZE]
x_val, y_val = x_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE], y_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
x_test, y_test = x_all[TRAIN_SIZE + VAL_SIZE:], y_all[TRAIN_SIZE + VAL_SIZE:]

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [ ]:
INPUT_SHAPE = (32, 32, 3)

In [ ]:
base_model_res = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=INPUT_SHAPE,
    pooling='avg'
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
x_res = base_model_res.output
predictions_res = Dense(num_classes, activation='softmax')(x_res)
model_res = Model(inputs=base_model_res.input, outputs=predictions_res)
base_model_res.trainable = True

In [ ]:
LEARNING_RATE = 1e-4
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
LOSS_FUNCTION = 'categorical_crossentropy'

In [ ]:
model_res.compile(
    optimizer=OPTIMIZER,
    loss=LOSS_FUNCTION,
    metrics=['accuracy']
)
print("--- ResNet50 Model Summary (for Parameter Count metric) ---")
model_res.summary()

--- ResNet50 Model Summary (for Parameter Count metric) ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 38, 38, 3) │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 16, 16,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 16, 16,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 16, 16,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 18, 18,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 8, 8, 64)  │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 8, 8, 64)  │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 8, 8, 64)  │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 8, 8, 64)  │          0 │ conv2_block1_1_b… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 8, 8, 64)  │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 8, 8, 64)  │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 8, 8, 64)  │          0 │ conv2_block1_2_b… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 8, 8, 256) │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 8, 8, 256) │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 8, 8, 256) │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 8, 8, 256) │      1,024 │ conv2_block1_3_c

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 23,555,082 (89.86 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
import time
print(f"Starting ResNet50 Training for {EPOCHS} epochs...")

history_res = model_res.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    verbose=1
)
print("\n--- ResNet50 Model Evaluation ---")
start_time_res = time.time()
loss_res, accuracy_res = model_res.evaluate(x_test, y_test, verbose=0)
end_time_res = time.time()
total_inference_time_res = end_time_res - start_time_res
time_per_image_res = total_inference_time_res / x_test.shape[0]
print(f"Final Test Accuracy (ResNet50): {accuracy_res*100:.2f}%")
print(f"Final Test Loss (ResNet50): {loss_res:.4f}")
print(f"Total Inference Time (6000 images): {total_inference_time_res:.4f} seconds")
print(f"Inference Time Per Image (ResNet50): {time_per_image_res*1000:.2f} ms")

Starting ResNet50 Training for 10 epochs...
Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 98s 32ms/step - accuracy: 0.4623 - loss: 1.7176 - val_accuracy: 0.7182 - val_loss: 0.8675
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.7422 - loss: 0.7634 - val_accuracy: 0.7673 - val_loss: 0.6979
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.8084 - loss: 0.5499 - val_accuracy: 0.7938 - val_loss: 0.6391
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.8607 - loss: 0.4147 - val_accuracy: 0.7963 - val_loss: 0.6590
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.8902 - loss: 0.3262 - val_accuracy: 0.8157 - val_loss: 0.6180
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.9184 - loss: 0.2419 - val_accuracy: 0.7938 - val_loss: 1.0778
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.9306 - loss: 0.2070 - val_accuracy: 0.8008 - val_loss: 0.8316
Epoch 8/10
1500/1500 ━━━━━━━━━━━